<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 19 · LLMs as Research and Coding Assistants

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals

This notebook demonstrates how Large Language Models (LLMs) can assist research
and coding workflows in a reproducible way:

- define reusable prompt templates,
- use a local mock LLM client so the notebook runs without API keys, and
- apply the client to summarize experiment logs and code snippets.


### Getting Help While Working with LLMs

- Chapter 19 provides the conceptual framing of assistants.
- Appendix F and project docs cover environment variables and secrets handling
  when you connect to real APIs.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

import json
import textwrap

### Data Loading
We keep the price panel available for quick illustrations.

In [ ]:
prices = pd.read_csv(DATA_PATH,
parse_dates=['Date']).set_index('Date').sort_index().ffill()

## 1. Prompt Template Library

We capture common LLM tasks as simple Python templates so they are
version-controlled and easy to reuse.


In [ ]:
PROMPTS = {
    'experiment_summary': textwrap.dedent('''
        You are an assistant for quantitative research. Summarize the
        following experiment metadata in 3 bullet points for a weekly lab
        meeting. Emphasize model type, data span, and key risk notes.

        Metadata: {metadata}
    '''),
    'code_review': textwrap.dedent('''
        You are helping a junior quant understand a code snippet.

        Explain what this code does and mention two potential pitfalls:

        {code}
    '''),
}

def render_prompt(name: str, **kwargs) -> str:
    template = PROMPTS[name]
    return template.format(**kwargs)

## 2. Mock LLM Client

To keep the notebook self-contained, we implement a small mock client that
returns a truncated echo of the prompt instead of calling a remote API.


In [ ]:
class MockLLMClient:
    def complete(self, prompt: str) -> str:
        preview = prompt.replace('', ' ')[:200]
        return '[MOCK LLM RESPONSE] ' + preview + ' ...'

llm = MockLLMClient()

## 3. Summarizing an Experiment Log

We load a JSON experiment log (if available) and ask the mock assistant to
produce a short human-readable summary.


In [ ]:
log_path = Path('../reports/model_risk_rf.json')
if log_path.exists():
    metadata = json.loads(log_path.read_text())
else:
    metadata = {
        'model': 'RandomForestRegressor',
        'train_start': '2020-01-01',
        'train_end': '2025-01-01',
    }
prompt = render_prompt('experiment_summary', metadata=json.dumps(metadata, indent=2))
print(llm.complete(prompt))

## 4. Code Explanation Example

The same client can explain nontrivial code blocks to speed up reviews and
onboarding.


In [ ]:
snippet = """prices = pd.read_csv(DATA_PATH,
parse_dates=['Date']).set_index('Date').sort_index().ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()
rolling_vol = log_rets.rolling(63).std() * np.sqrt(252)
"""

prompt = render_prompt('code_review', code=snippet)
print(llm.complete(prompt))

## 5. Exercises

### Exercise 1 – Custom Prompt
Design a template that drafts a risk memo from a performance table.
<details><summary>Hint</summary>Include placeholders for horizon, Sharpe, and
max drawdown.</details>

### Exercise 2 – Notebook Summary
Write a helper that collects all headings in a notebook and feeds them to the
LLM for a one-paragraph summary.
<details><summary>Hint</summary>Use nbformat to read cell sources and filter
markdown headings.</details>

### Exercise 3 – Safety Checklist
Create a prompt that asks the LLM to flag any compliance risks in a strategy
description.
<details><summary>Hint</summary>Give the model a short checklist (no leverage
limits breached, no restricted instruments, etc.).</details>


## 6. Takeaways

LLM assistants become most useful when their prompts are treated as
first-class, version-controlled artifacts that integrate with your existing
logs, metrics, and notebooks.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">